[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/23_cross_attention.ipynb)

# 🟠 Medium: Multi-Head Cross-Attention

Implement **multi-head cross-attention** (encoder-decoder attention).

### Signature
```python
class MultiHeadCrossAttention(nn.Module):
    def __init__(self, d_model: int, num_heads: int): ...
    def forward(self, x_q: Tensor, x_kv: Tensor) -> Tensor:
        # x_q: (B, S_q, D) — decoder queries
        # x_kv: (B, S_kv, D) — encoder keys/values
```

### Key Differences from Self-Attention
- Q comes from the decoder, K and V come from the encoder
- No causal mask (all encoder positions visible)

In [1]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 3.3 MB/s eta 0:00:00


In [2]:
import torch
import torch.nn as nn
import math

In [7]:
# ✏️ YOUR IMPLEMENTATION HERE

class MultiHeadCrossAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        self.d_k=d_model//num_heads
        self.num_heads=num_heads
        self.d_model=d_model
        self.W_q=nn.Linear(d_model,d_model)
        self.W_k=nn.Linear(d_model,d_model)
        self.W_v=nn.Linear(d_model,d_model)
        self.W_o=nn.Linear(d_model,d_model)


    def forward(self, x_q, x_kv):
        B,s_q,_=x_q.shape
        B,s_kv,_=x_kv.shape

        q=self.W_q(x_q)
        k=self.W_k(x_kv)
        v=self.W_v(x_kv)

        q=q.view(B,-1,self.num_heads,self.d_k).transpose(1,2)
        k=k.view(B,-1,self.num_heads,self.d_k).transpose(1,2)
        v=v.view(B,-1,self.num_heads,self.d_k).transpose(1,2)

        scores=torch.matmul(q,k.transpose(-1,-2))/math.sqrt(self.d_k)
        attention=torch.softmax(scores,dim=-1)

        context=torch.matmul(attention,v)
        context=context.transpose(1,2).contiguous().view(B,-1,self.d_model)

        output=self.W_o(context)
        return output






In [8]:
# 🧪 Debug
attn = MultiHeadCrossAttention(64, 4)
x_q = torch.randn(2, 6, 64)
x_kv = torch.randn(2, 10, 64)
print('Output:', attn(x_q, x_kv).shape)

Output: torch.Size([2, 6, 64])


In [9]:
# ✅ SUBMIT
from torch_judge import check
check('cross_attention')


🧪 Testing: Multi-Head Cross-Attention (Medium)
──────────────────────────────────────────────────
  ✅ [1/4] Output shape (2.3ms)
  ✅ [2/4] Q and KV different lengths (1.8ms)
  ✅ [3/4] No causal mask — all KV affects all Q (54.8ms)
  ✅ [4/4] Gradient flow (34.7ms)
──────────────────────────────────────────────────
  🎉 All 4 tests passed! (93.6ms total)
  Progress saved. Run status() to see your dashboard.

